# 07 — OE6: Triangulación (mercado global vs mercado chileno)

Contrasta **PI6** y **H7**: ¿difiere la transmisión del impacto macro entre el mercado global
(muestra B) y el mercado chileno (muestras A y C)?

Requisito: haber construido `panel_B/A/C.parquet` y `series_B/A/C.parquet` (notebook 00,
una vez por muestra).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src import config as C
from src import build_panel as bp
from src import triangulation as tri

## 1. (Re)construir los tres paneles
Si ya los construiste en el notebook 00, salta esta celda.

In [ ]:
for suf, uni in [('_B', C.UNIVERSO_PRINCIPAL),
                 ('_A', C.UNIVERSO_LOCAL_COBRE),
                 ('_C', C.UNIVERSO_LOCAL_MINERIA)]:
    bp.construir(guardar=True, universo=uni, sufijo=suf)

## 2. Tabla comparativa (OE6)
Sensibilidad al cobre, R² y fracción de varianza explicada por shocks globales (FEVD), por muestra.

In [ ]:
tabla = tri.comparar_muestras()
display(tabla.round(4))
tabla.to_csv(C.TABLES / 'oe6_comparacion_mercados.csv')

## 3. Visualización comparativa

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
tabla['beta_cobre'].plot.bar(ax=ax[0], color=['steelblue','firebrick','salmon'])
ax[0].set_title('Sensibilidad al cobre (β)'); ax[0].set_xlabel('')
tabla['FEVD_global_h12'].plot.bar(ax=ax[1], color=['steelblue','firebrick','salmon'])
ax[1].set_title('Varianza explicada por shocks globales (h=12)'); ax[1].set_xlabel('')
plt.tight_layout(); plt.savefig(C.FIGURES / 'oe6_triangulacion.png', dpi=120)

## 4. Lectura (OE6 / H7)
Compara entre muestras:
- **β del cobre**: ¿es similar entre cobre puro global (B) y local (A)? ¿menor en la minería
  mixta (C, por incluir hierro/litio/molibdeno)?
- **FEVD global**: ¿el mercado global (B) atribuye mayor fracción de varianza a shocks globales
  que el chileno (A/C)? Eso apoyaría **H7** (transmisión más completa en el mercado global).

> Recuerda: la lectura definitiva requiere las series locales del BCCh (TPM, IMACEC, EMBI) y los
> controles de empresa. Sin ellas, esta tabla valida el método pero no es interpretable como
> resultado final.